In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false; // disable scroll bar when displaying Folium map
}

# Assignment 2

Before working on this assignment please read these instructions fully. In the submission area, you will notice that you can click the link to **Preview the Grading** for each step of the assignment. This is the criteria that will be used for peer grading. Please familiarize yourself with the criteria before beginning the assignment.

The data for this assignment comes from a subset of The National Centers for Environmental Information (NCEI) [Global Historical Climatology Network daily (GHCNd)](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily) (GHCN-Daily). The GHCN-Daily is comprised of daily climate records from thousands of land surface stations across the globe - it's a wonderfully large dataset to play with! In particular, you will be asked to use data from the Ann Arbor Michigan location (my home!). and this is stored in the file: `assets/fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv`

Each row in this datafile corresponds to a single observation from a weather station, and has the following variables:
* **id** : station identification code
* **date** : date in YYYY-MM-DD format (e.g. 2012-01-24 = January 24, 2012)
* **element** : indicator of element type
    * TMAX : Maximum temperature (tenths of degrees C)
    * TMIN : Minimum temperature (tenths of degrees C)
* **value** : data value for element (tenths of degrees C)

For this assignment, you must:

1. Read the documentation and familiarize yourself with the dataset, then write a python notebook which plots line graphs of the record high and record low temperatures by day of the year over the period 2005-2014. The area between the record high and record low temperatures for each day should be shaded.
2. Overlay a scatter of the 2015 data for any points (highs and lows) for which the ten year record (2005-2014) record high or record low was broken in 2015. (Based on the graph, do you think extreme weather is getting more frequent in 2015?)
3. Watch out for leap days (i.e. February 29th), it is reasonable to remove these points from the dataset for the purpose of this visualization.
4. Make the visual nice! Leverage principles from the first module in this course when developing your solution. Consider issues such as legends, labels, and chart junk.

I've written some steps I think would be good to go through, but there are other ways to solve this assignment so feel free to explore the pandas library! What I really want to see is an image that looks like this sketch I drew at my desk:

![](assets/chris_sketch.png)

In [1]:
#  I'll be using the folium package to render the data into a map in Jupyter.

import folium
import pandas as pd

# get the location information for this dataset
df = pd.read_csv('assets/BinSize_d400.csv')
station_locations_by_hash = df[df['hash'] == 'fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89']

# get longitude and lattitude to plot
lons = station_locations_by_hash['LONGITUDE'].tolist()
lats = station_locations_by_hash['LATITUDE'].tolist()

# plot on a beautiful folium map
my_map = folium.Map(location = [lats[0], lons[0]], height = 500,  zoom_start = 9)
for lat, lon in zip(lats, lons):
    folium.Marker([lat, lon]).add_to(my_map)

# render map in Jupyter
display(my_map)

## Step 1
Load the dataset and transform the data into Celsius (refer to documentation) then extract all of the rows which have minimum or maximum temperatures.

__hint: when I did this step I had two DataFrame objects, each with ~80,000 entries in it__

In [2]:
import pandas as pd
df = pd.read_csv('assets/fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv')
df.head()

,ID,Date,Element,Data_Value
0,USW00094889,2014-11-12,TMAX,22
1,USC00208972,2009-04-29,TMIN,56
2,USC00200032,2008-05-26,TMAX,278
3,USC00205563,2005-11-11,TMAX,139
4,USC00200230,2014-02-27,TMAX,-106


In [3]:
# In this code cell, transform the Data_Value column
#transform the data into Celsius
print(df.head())
df['Data_Value'] = df['Data_Value'].apply(lambda x: (x-32)/1.8)
print(df.head())
#then extract all of the rows which have minimum or maximum temperatures
df_TMIN= df
df_TMIN = df[df['Element'] == 'TMIN']

df_TMAX= df
df_TMAX = df[df['Element'] == 'TMAX']

#print(df_TMAX.shape)
#print(df_TMIN.shape)

            ID        Date Element  Data_Value
0  USW00094889  2014-11-12    TMAX          22
1  USC00208972  2009-04-29    TMIN          56
2  USC00200032  2008-05-26    TMAX         278
3  USC00205563  2005-11-11    TMAX         139
4  USC00200230  2014-02-27    TMAX        -106
            ID        Date Element  Data_Value
0  USW00094889  2014-11-12    TMAX   -5.555556
1  USC00208972  2009-04-29    TMIN   13.333333
2  USC00200032  2008-05-26    TMAX  136.666667
3  USC00205563  2005-11-11    TMAX   59.444444
4  USC00200230  2014-02-27    TMAX  -76.666667


## Step 2
In order to visualize the data we would plot the min and max data for each day of the year between the years 2005 and 2014 across all weather stations. But we also need to find out when the min or max temperature in 2015 falls below the min or rises above the max for the previous decade.

If you did step 1 you have two Series objects with min and max times for the years 2005 through 2015. You can use Pandas `groupby` to create max and min temperature Series objects across all weather stations for each day of these years, and you can deal with the records for February 29 (the leap year) by dropping them.

__hint: when I finished this step, I had two DataFrame objects, each with exactly 4015 observations in them__

In [4]:
#unique_values = df_TMAX['Date'].unique()
#print(len(unique_values))

#drop records for Feb 29
df_TMAX = df_TMAX[~df_TMAX['Date'].str.contains(r'\d{4}-02-29', na=False)]
df_TMIN = df_TMIN[~df_TMIN['Date'].str.contains(r'\d{4}-02-29', na=False)]
#print(df_TMAX.shape)
#print(df_TMIN.shape)
# create a DataFrame of maximum temperature by date
series_TMAX=df_TMAX.groupby('Date').agg(['max'])
#print(series_TMAX.head())
# create a DataFrame of minimum temperatures by date
series_TMIN=df_TMIN.groupby('Date').agg(['min'])
#print(series_TMIN)

## Step 3
Now that you have grouped the daily max and min temperatures for each day of the years 2005 through 2015, you can separate out the data for 2015. Then you can use the Pandas `groupby` function to find the max and min of the temperature data for each __day of the year__ for the 2005-2014 data.

__hint: at the end of this step I had two DataFrames, one of maximum and the other of minimum values, which each had 365 observations in them. I also had another pair of similar DataFrames but only for the year 2015.__

In [5]:
#seperate out data for 2015
series_TMAX_2015=series_TMAX[series_TMAX.index.str.match(r'2015-\d{2}-\d{2}')]
#print(series_TMAX_2015)
series_TMIN_2015=series_TMIN[series_TMIN.index.str.match(r'2015-\d{2}-\d{2}')]
#print(series_TMIN_2015)

# calculate the minimum and maximum values for the day of the year for 2005 through 2014
df_TMAX = pd.DataFrame(series_TMAX)
df_TMAX.columns = ['ID', 'Element', 'Data_Value']
df_TMAX = df_TMAX[~df_TMAX.index.str.contains(r'2015-\d{2}-\d{2}', na=False)]
#print(df_TMAX)
df_TMAX['Date'] = df_TMAX.index
df_TMAX_05to14 = df_TMAX[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
df_TMAX_day=df_TMAX_05to14.groupby(df_TMAX_05to14['Date'].str[5:]).max()
#print(df_TMAX_day.head(10))


df_TMIN = pd.DataFrame(series_TMIN)
df_TMIN.columns = ['ID', 'Element', 'Data_Value']
df_TMIN = df_TMIN[~df_TMIN.index.str.contains(r'2015-\d{2}-\d{2}', na=False)]
#print(df_TMIN.tail())
df_TMIN['Date'] = df_TMIN.index
df_TMIN_05to14 = df_TMIN[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
df_TMIN_day=df_TMIN_05to14.groupby(df_TMIN_05to14['Date'].str[5:]).min()
#print(df_TMIN_05to14.groupby(df_TMIN_05to14['Date'].str[5:]).get_group('01-01'))
#print(df_TMIN_day.head(10))


# calculate the minimum and maximum values for the years 2015
df_TMAX_2015 = pd.DataFrame(series_TMAX_2015)
df_TMAX_2015.columns = ['ID', 'Element', 'Data_Value']
df_TMAX_2015['Date'] = df_TMAX_2015.index
df_TMAX_15 = df_TMAX_2015[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
#print(df_TMAX_15)

df_TMIN_2015 = pd.DataFrame(series_TMIN_2015)
df_TMIN_2015.columns = ['ID', 'Element', 'Data_Value']
df_TMIN_2015['Date'] = df_TMIN_2015.index
df_TMIN_15 = df_TMIN_2015[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
#print(df_TMIN_15)




## Step 4
Now it's time to plot! You need to explore matplotlib in order to plot line graphs of the min and max temperatures for the years 2005 through 2014 and to scatter plot __only__ the daily 2015 temperatures that exceeded those values.

In [17]:
import matplotlib.pyplot as plt
from calendar import month_abbr

# put your plotting code here!
plt.rcParams['xtick.labelsize'] = 150
plt.rcParams['ytick.labelsize'] = 150


#print(df_TMAX_day)
lst_xlabels=[]
for date in df_TMAX_day['Date']:
    #print(date)
    if date[-2:] == '15':
        #print(date)
        #print(month_abbr[int(date[5:7])])
        lst_xlabels.append(month_abbr[int(date[5:7])])
    else:
        lst_xlabels.append('')
        
plt.figure(figsize=(365, 100))
plt.plot(df_TMAX_day['Date'],df_TMAX_day['Data_Value'], label="Maximum daily temparature from 2005-2014", lw=20)
plt.plot(df_TMAX_day['Date'],df_TMIN_day['Data_Value'], label="Minimum daily temparature from 2005-2014", lw=20)

plt.xticks(df_TMAX_day['Date'], lst_xlabels)

ax = plt.gca()
ax.set_xlabel('Month', fontsize=300)
ax.set_ylabel('Temperature (Celsius)', fontsize=300)
ax.set_title("Comparison of daily Temperature Variation: 2015 vs past decade", fontsize=300)
  


plt.gca().fill_between(df_TMAX_day['Date'], 
                       df_TMAX_day['Data_Value'], df_TMIN_day['Data_Value'], 
                       facecolor='green', 
                       alpha=0.1)

#scatter plot only the daily 2015 temperatures that exceeded those values

#print(df_TMAX_15.head(10))
df_TMAX=df_TMAX_day[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
#print(df_TMAX.head(10))

df_TMAX_15['Data_Value gt']= df_TMAX_15['Data_Value'] - df_TMAX['Data_Value']
df_TMAX_15_gt=df_TMAX_15[df_TMAX_15['Data_Value gt'] > 0]
#replace 2015 by 2014 to match x-cordinates
df_TMAX_15_gt['Date'] = df_TMAX_15_gt['Date'].astype(str)
df_TMAX_15_gt['Date'] = df_TMAX_15_gt['Date'].str.replace('2015', '2014', regex=False)

#print(df_TMAX_15_gt['Date'].dtype)
#print(df_TMAX_15_gt)

plt.scatter(df_TMAX_15_gt['Date'], df_TMAX_15_gt['Data_Value'], color='red', marker='x', s=5000, linewidths=20, label='temp in 2015 above that over past decade')


df_TMIN=df_TMIN_day[['Date', 'ID', 'Element','Data_Value']].reset_index(drop=True)
#print(df_TMIN.head(10))
#print(df_TMIN_15)
df_TMIN_15['Data_Value lt']= df_TMIN_15['Data_Value'] - df_TMIN['Data_Value']
df_TMIN_15_lt=df_TMIN_15[df_TMIN_15['Data_Value lt'] < 0]
df_TMIN_15_lt['Date'] = df_TMIN_15_lt['Date'].astype(str)
df_TMIN_15_lt['Date'] = df_TMIN_15_lt['Date'].str.replace('2015', '2014', regex=False)

plt.scatter(df_TMIN_15_lt['Date'], df_TMIN_15_lt['Data_Value'], color='green', marker='x', s=5000, linewidths=20, label='temp in 2015 below that over past decade')

plt.legend(fontsize=150)
